In [1]:
import pandas as pd
import re
import numpy as np
from collections import defaultdict
from analysis_util import fisher_metrics_with_optional_bootstrap, project_root, filter_by_fisher, top_n_models_by_fisher

ROOT_DIR = project_root()

# print(f"Project root directory: {ROOT_DIR}")


In [2]:
df = pd.read_csv(
    ROOT_DIR / "data/model_grades/criminal_law/criminal_law_ground_truth_and_model_output.csv"
)

In [3]:

_COL_RE = re.compile(
    r"^(?P<engine>[^_]+)_prompt_(?P<prompt_id>.+?)__"
    r"(?P<model>.+?)__i(?P<iter>\d+)__"
    r"(?P<hash>h[0-9a-f]+|hc[0-9a-f]+)__"
    r"(?P<kind>.+)$"
)

def parse_prompt_col(col: str) -> dict:
    m = _COL_RE.match(col)
    if not m:
        raise ValueError(f"Unexpected column format: {col}")
    d = m.groupdict()
    d["iter"] = int(d["iter"])
    d["prompt_family"] = f"prompt_{d['prompt_id']}"  # e.g. prompt_v1_ta_min_ha
    return d


def build_columns_registry(
    all_cols: list[str],
    *,
    skip_iters: set[int] = {4, 5},
    require_iters: tuple[int, ...] | None = (1, 2, 3),
    include_engine_in_key: bool = False,
) -> dict:
    """
    Returns:
      if include_engine_in_key:
        { (engine, prompt_family, model): [col_i1, col_i2, col_i3] }
      else:
        { (prompt_family, model): [col_i1, col_i2, col_i3] }
    """
    tmp = defaultdict(dict)

    for col in all_cols:
        d = parse_prompt_col(col)
        if d["iter"] in skip_iters:
            continue

        key = (d["engine"], d["prompt_family"], d["model"]) if include_engine_in_key else (d["prompt_family"], d["model"])
        tmp[key][d["iter"]] = col

    out = {}
    for key, by_iter in tmp.items():
        if require_iters is not None:
            missing = [i for i in require_iters if i not in by_iter]
            if missing:
                # skips partial sets like the v4_1_ts_ext_model_ha entries that only have i1
                continue
        out[key] = [by_iter[i] for i in sorted(by_iter)]
    return out


all_prompt_cols = df.filter(like="__extracted_element").columns.tolist()

COLUMNS = build_columns_registry(
    all_prompt_cols,
    require_iters=(1, 2, 3),   # set to None if you want to keep partial sets
    include_engine_in_key=True,  # True only if you expect engine collisions
)


def run_eval_suite(
    df,
    *,
    columns_registry: dict,
    gold_col: str,
    eval_fn,
):
    results = {}  # key -> result dict
    for key, cols in columns_registry.items():
        results[key] = eval_fn(df, rater_cols=cols, reference_col=gold_col)
    return results

gold_col = "points"

results = run_eval_suite(
    df,
    columns_registry=COLUMNS,
    gold_col=gold_col,
    eval_fn=fisher_metrics_with_optional_bootstrap,
)



def results_to_tidy_df(results: dict) -> pd.DataFrame:
    rows = []
    for key, r in results.items():
        # key is (prompt_family, model) OR (engine, prompt_family, model)
        if len(key) == 2:
            prompt_family, model = key
            engine = None
        else:
            engine, prompt_family, model = key

        label = f"{model}_{prompt_family}"

        if engine is not None:
            # optional: keep engine visible in label
            label = f"{engine}:{label}"

        rows.extend([
            {
                "model": label,
                "prompt_family": prompt_family,
                "base_model": model,
                "engine": engine,
                "metric": "qwk",
                "mean_fisher": r["point_mean_qwk_fisher"],
                "sd_raw": r["point_sd_qwk_across_runs_kappa_space"],
            },
            {
                "model": label,
                "prompt_family": prompt_family,
                "base_model": model,
                "engine": engine,
                "metric": "pearson",
                "mean_fisher": r["point_mean_pearson_fisher"],
                "sd_raw": r["point_sd_pearson_across_runs_raw_space"],
            },
            {
                "model": label,
                "prompt_family": prompt_family,
                "base_model": model,
                "engine": engine,
                "metric": "spearman",
                "mean_fisher": r["point_mean_spearman_fisher"],
                "sd_raw": r["point_sd_spearman_across_runs_raw_space"],
            },
        ])

    return pd.DataFrame(rows)

tidy_df = results_to_tidy_df(results)


In [4]:
# tidy_df

which_metric = "qwk" # "pearson"  # "spearman" # "qwk"


df_f_f = (
    tidy_df.query(f'metric == "{which_metric}"')
           .sort_values(["prompt_family", "mean_fisher"], ascending=[True, False])
           .loc[:, ["prompt_family", "base_model", "mean_fisher", "sd_raw"]]
           .round(3)
)
display(df_f_f)


,prompt_family,base_model,mean_fisher,sd_raw
24,prompt_v1_ta_min_ha,gpt-5.2-2025-12-11,0.311,0.049
0,prompt_v1_ta_min_ha,gpt-5-2025-08-07,0.309,0.017
18,prompt_v1_ta_min_ha,gpt-5.1-2025-11-13,0.181,0.016
36,prompt_v1_ta_min_ha,gemini-2.5-pro,0.161,0.030
108,prompt_v1_ta_min_ha,gpt-5-mini-2025-08-07,0.097,0.024
39,prompt_v1_ta_min_ha,deepseek-v3.1-terminus,0.088,0.046
102,prompt_v1_ta_min_ha,mistral-large-2512,0.067,0.004
72,prompt_v1_ta_min_ha,gpt-oss-20b,0.065,0.098
30,prompt_v1_ta_min_ha,gpt-4o-2024-11-20,0.063,0.005
12,prompt_v1_ta_min_ha,gpt-4.1-2025-04-14,0.052,0.008


In [6]:
###
### Random baseline
###


# Reproducible RNG
rng = np.random.default_rng(seed=42)

# Generate 5 random baseline columns
n_baselines = 3
low, high = 0, 19  # high is exclusive

random_baselines = rng.integers(
    low=low,
    high=high,
    size=(len(df), n_baselines)
)

# Assign to DataFrame
for i in range(n_baselines):
    df[f"random_baseline_{i+1}"] = random_baselines[:, i]



random_baselines_col = ['random_baseline_1',
                        'random_baseline_2',
                        'random_baseline_3']

random_baseline=fisher_metrics_with_optional_bootstrap(df, rater_cols=random_baselines_col, reference_col=gold_col)
random_baseline

{'point_mean_qwk_fisher': -0.052657719756291126,
 'point_sd_qwk_across_runs_kappa_space': 0.2595924934348788,
 'point_mean_pearson_fisher': -0.06845078565067574,
 'point_sd_pearson_across_runs_raw_space': 0.3070723832161212,
 'point_mean_spearman_fisher': -0.06642222231617478,
 'point_sd_spearman_across_runs_raw_space': 0.27895741214962155}

In [ ]:
###
### Ovelap analysis between qwk and pearson top models
###

In [25]:
###
### N = 5
###


###
### Top-n models by perason and qwk. Taking the highest score if same model apepar in both lists and showing the corresponding prompt_family.
###




print("Top models by Pearson Fisher:")
display(top_n_models_by_fisher(tidy_df.query('metric == "pearson"').sort_values(["prompt_family", "mean_fisher"], ascending=[True, False]).loc[:, ["prompt_family", "base_model", "mean_fisher", "sd_raw"]].round(3), n=5))


print("Top models by QWK Fisher:")
display(top_n_models_by_fisher(tidy_df.query('metric == "qwk"').sort_values(["prompt_family", "mean_fisher"], ascending=[True, False]).loc[:, ["prompt_family", "base_model", "mean_fisher", "sd_raw"]].round(3), n=5))

Top models by Pearson Fisher:


,prompt_family,base_model,mean_fisher,sd_raw
127,prompt_v5_ts_ext_rubric_model_ha,gpt-5-2025-08-07,0.599,0.005
163,prompt_v5_ts_ext_rubric_model_ha,gpt-5.2-2025-12-11,0.589,0.023
214,prompt_v3_ts_ext_rubric_ha,gpt-4.1-2025-04-14,0.581,0.030
133,prompt_v5_ts_ext_rubric_model_ha,gemini-2.5-pro,0.574,0.011
199,prompt_v5_ts_ext_rubric_model_ha,gpt-5-mini-2025-08-07,0.569,0.009


Top models by QWK Fisher:


,prompt_family,base_model,mean_fisher,sd_raw
126,prompt_v5_ts_ext_rubric_model_ha,gpt-5-2025-08-07,0.599,0.005
132,prompt_v5_ts_ext_rubric_model_ha,gemini-2.5-pro,0.565,0.015
195,prompt_v3_ts_ext_rubric_ha,gpt-5-mini-2025-08-07,0.499,0.047
216,prompt_v5_ts_ext_rubric_model_ha,gpt-4.1-2025-04-14,0.478,0.025
27,prompt_v4_ts_ext_model_ha,gpt-5.2-2025-12-11,0.465,0.036


In [26]:
###
### Overlap models between top Pearson and QWK
###


qwk_top_n = top_n_models_by_fisher(
    tidy_df.query('metric == "qwk"')
           .sort_values(["prompt_family", "mean_fisher"], ascending=[True, False])
           .loc[:, ["prompt_family", "base_model", "mean_fisher", "sd_raw"]]
           .round(3),
    n=5
)

pearson_top_n = top_n_models_by_fisher(
    tidy_df.query('metric == "pearson"')
           .sort_values(["prompt_family", "mean_fisher"], ascending=[True, False])
           .loc[:, ["prompt_family", "base_model", "mean_fisher", "sd_raw"]]
           .round(3),
    n=5
)

overlap_models = sorted(
    set(qwk_top_n["base_model"]) & set(pearson_top_n["base_model"])
)

print(f"Number of overlapping models: {len(overlap_models)}")
print("Overlapping model names:")
for model in overlap_models:
    print(model)

Number of overlapping models: 5
Overlapping model names:
gemini-2.5-pro
gpt-4.1-2025-04-14
gpt-5-2025-08-07
gpt-5-mini-2025-08-07
gpt-5.2-2025-12-11


### Extras

In [ ]:
###
### If you want to export the table to LaTeX, you can use the following code.
###

# print(df_f_f.to_latex(index=False, float_format="%.3f", escape=True, longtable=True, caption=f"Evaluation results for {which_metric.upper()} metric. Criminal Law domain."))
# print(df_f_f.to_latex(index=False, float_format="%.3f", escape=True, longtable=True, caption=f"Evaluation results for {which_metric.upper()} metric. Criminal Law domain."))


# display(filter_by_fisher(df_f_f, threshold=0.464))